In [1]:
import numpy as np
import itertools
import axelrod
from axelrod.action import Action, actions_to_str
from axelrod.player import Player
from axelrod.strategy_transformers import (
    FinalTransformer,
    TrackHistoryTransformer,
)
import skfuzzy as fuzz
from skfuzzy import control as ctrl
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.core.problem import Problem
from pymoo.core.sampling import Sampling
from pymoo.core.crossover import Crossover
from pymoo.core.mutation import Mutation
from pymoo.optimize import minimize
from pymoo.visualization.scatter import Scatter
from pymoo.operators.sampling.rnd import BinaryRandomSampling
from pymoo.operators.crossover.ux import UniformCrossover
from pymoo.operators.mutation.bitflip import BitflipMutation
from collections import Counter
from math import exp
from pymoo.core.sampling import Sampling

In [2]:
C, D = Action.C, Action.D

class FuzzyMethods():
    @staticmethod
    def calc_cooperation(self, opponent):
        return (Counter(opponent.history)[C])/len(opponent.history)*100
    
    @staticmethod
    def calc_adaptivity(self, opponent):
        adapCounter = 0
        adapReaction = 0

        if(len(self.history) < 3): 
            return 0
        
        for i in range(3, len(self.history)):
            if (self.history[i-3] == C and self.history[i-2] == D):
                adapCounter += 1
                if (opponent.history[i-1] == D):
                    adapReaction += 1
                elif (self.history[i-1] == D and opponent.history[i] == D):
                    adapReaction += 0.5
            elif (self.history[i-3] == D and self.history[i-2] == C):
                adapCounter += 1
                if (opponent.history[i-1] == C):
                    adapReaction += 1
                elif (self.history[i-1] == C and opponent.history[i] == C):
                    adapReaction += 0.5

        if adapCounter == 0:
            return 0
        
        return adapReaction/adapCounter*100

    
    @staticmethod
    def calc_forgiveness(self, opponent):
        DCounter = 0
        punishmentCounter = 0

        for i in range(0, len(self.history)-1):
            if(self.history[i] == D):
                DCounter += 1
                for j in range (i+1, len(opponent.history)):
                    if(opponent.history[j] == C):
                        break
                    else:
                        punishmentCounter += 1

  
        if punishmentCounter > 0:
            return DCounter/punishmentCounter*100
        else:
            return 100
    
    @staticmethod
    def calc_stochastic(self, opponent):
        patterns = [
            [C, C, C],
            [C, C, D],
            [C, D, C],
            [C, D, D],
            [D, C, C],
            [D, C, D],
            [D, D, C],
            [D, D, D]
        ]

        nonStochasticCounter = 0
        patternPlayedCounter = 0

        for p in patterns:
            opponentsReactions = []
            for i in range(0, len(self.history)-3):
                if ([self.history[i], self.history[i+1], self.history[i+2]] == p):
                    opponentsReactions.append([opponent.history[i+1], opponent.history[i+2], opponent.history[i+3]])
            
            unique_patterns = len(set(tuple(sub) for sub in opponentsReactions))

            if(len(opponentsReactions) > 0):
                nonStochasticCounter += 0 if unique_patterns == 1 else unique_patterns
                patternPlayedCounter += len(opponentsReactions)
        
        if patternPlayedCounter == 0:
            return 0
        
        return nonStochasticCounter/patternPlayedCounter*100
    

    @staticmethod
    def sigmoid(x, center, scale=10):
        return 1 / (1 + exp(-scale * (x - center)))
    
    @staticmethod
    def fuzzy_gate(mu_D, mu_C, d_center = 0.4, c_center = 0.6):
        d_condition = FuzzyMethods.sigmoid(mu_D, center=d_center)
        c_condition = 1 - FuzzyMethods.sigmoid(mu_C, center=c_center)

        w1 = d_condition * c_condition
        w2 = 1 - w1

        z1 = 1
        z2 = 0

        z = (w1 * z1 + w2 * z2) / (w1 + w2 + 1e-6)
        return z

In [3]:

# ─────────────────────────────────────────────
# GENERISANJE SVIH MOGUĆIH PRAVILA
# ─────────────────────────────────────────────

# None znači da input nije uključen u pravilo
ADAPTIVITY_TERMS  = [None, 'no', 'yes']
COOPERATION_TERMS = [None, 'low', 'medium', 'high']
FORGIVENESS_TERMS = [None, 'low', 'medium', 'high']
STOCHASTIC_TERMS  = [None, 'none', 'sometimes', 'always']
CONSEQUENTS       = ['D', 'C']

ALL_RULES = []
for adap, coop, forg, stoch, cons in itertools.product(
    ADAPTIVITY_TERMS,
    COOPERATION_TERMS,
    FORGIVENESS_TERMS,
    STOCHASTIC_TERMS,
    CONSEQUENTS
):
    # Mora imati bar jedan antecedent
    if adap is None and coop is None and forg is None and stoch is None:
        continue
    ALL_RULES.append((adap, coop, forg, stoch, cons))

N_RULES = len(ALL_RULES)
print(f"Ukupno mogućih pravila: {N_RULES}")






Ukupno mogućih pravila: 382


In [4]:
def build_player_from_binary(binary_vector):
    active_rules = [ALL_RULES[i] for i, active in enumerate(binary_vector) if active == 1]

    if len(active_rules) == 0:
        return None

    _cooperation = ctrl.Antecedent(np.arange(0, 100, 1), 'cooperation')
    _adaptivity  = ctrl.Antecedent(np.arange(0, 100, 1), 'adaptivity')
    _forgiveness = ctrl.Antecedent(np.arange(0, 100, 1), 'forgiveness')
    _stochastic  = ctrl.Antecedent(np.arange(0, 100, 1), 'stochastic')

    _cooperation.automf(names=["low", "medium", "high"])
    _adaptivity.automf(names=["no", "yes"])
    _forgiveness.automf(names=["low", "medium", "high"])
    _forgiveness['low']    = fuzz.gaussmf(_forgiveness.universe, 0, 25)
    _forgiveness['medium'] = fuzz.trimf(_forgiveness.universe, [25, 50, 75])
    _stochastic.automf(names=["none", "sometimes", "always"])

    _resulting_strategy = ctrl.Consequent(np.arange(0, 100, 1), 'resulting_strategy')
    _resulting_strategy['D'] = fuzz.trimf(_resulting_strategy.universe, [0,  25, 50])
    _resulting_strategy['C'] = fuzz.trimf(_resulting_strategy.universe, [35, 75, 99])

    # Prikupi koji inputi su zaista korišćeni u aktivnim pravilima
    used_inputs = set()
    for adap, coop, forg, stoch, cons in active_rules:
        if adap  is not None: used_inputs.add('adaptivity')
        if coop  is not None: used_inputs.add('cooperation')
        if forg  is not None: used_inputs.add('forgiveness')
        if stoch is not None: used_inputs.add('stochastic')

    rules = []
    for adap, coop, forg, stoch, cons in active_rules:
        antecedents = []
        if adap  is not None: antecedents.append(_adaptivity[adap])
        if coop  is not None: antecedents.append(_cooperation[coop])
        if forg  is not None: antecedents.append(_forgiveness[forg])
        if stoch is not None: antecedents.append(_stochastic[stoch])

        combined = antecedents[0]
        for ant in antecedents[1:]:
            combined = combined & ant

        rules.append(ctrl.Rule(combined, _resulting_strategy[cons]))

    rule_default = ctrl.Rule(
        _cooperation['low'] | _cooperation['medium'] | _cooperation['high'],
        _resulting_strategy['C']
    )
    rules.append(rule_default)

    try:
        _chosen_strategy = ctrl.ControlSystemSimulation(
            ctrl.ControlSystem(rules)
        )
    except Exception:
        return None

    _rs          = _resulting_strategy
    _cs          = _chosen_strategy
    _used        = used_inputs

    class MOEAFuzzy(Player):
        name = "MOEAFuzzy"
        classifier = {"memory_depth": float("inf"), "stochastic": False,
                      "long_run_time": False, "inspects_source": False,
                      "manipulates_source": False, "manipulates_state": False}

        resulting_strategy = _rs
        chosen_strategy    = _cs
        used_inputs        = _used
        first_time         = True
        h                  = {'Name': '', 'Fuzzy': [], 'Opponent': []}

        def strategy(self, opponent: Player) -> Action:
            if len(self.history) == 0 or D not in opponent.history:
                return C

            coop  = FuzzyMethods.calc_cooperation(self, opponent)
            adap  = FuzzyMethods.calc_adaptivity(self, opponent)
            forg  = FuzzyMethods.calc_forgiveness(self, opponent)
            stoch = FuzzyMethods.calc_stochastic(self, opponent)

            if 'cooperation' in self.used_inputs:
                self.chosen_strategy.input['cooperation'] = coop
            if 'adaptivity'  in self.used_inputs:
                self.chosen_strategy.input['adaptivity']  = adap
            if 'forgiveness' in self.used_inputs:
                self.chosen_strategy.input['forgiveness'] = forg
            if 'stochastic'  in self.used_inputs:
                self.chosen_strategy.input['stochastic']  = stoch

            try:
                self.chosen_strategy.compute()
                output_val   = self.chosen_strategy.output['resulting_strategy']
                d_membership = fuzz.interp_membership(self.resulting_strategy.universe, self.resulting_strategy['D'].mf, output_val)
                c_membership = fuzz.interp_membership(self.resulting_strategy.universe, self.resulting_strategy['C'].mf, output_val)
                if d_membership >= 0.4 and c_membership < 0.6:
                    return D
            except:
                return C
            return C

    return MOEAFuzzy()

In [5]:
def evaluate_binary(binary_vector):
    """Vraća (normalized_score, broj_aktivnih_pravila)."""
    n_active = int(sum(binary_vector))

    if n_active == 0:
        return 0.0, 0

    try:
        player    = build_player_from_binary(binary_vector)
        if player is None:
            return 0.0, n_active
        opponents = [s() for s in axelrod.stewart_plotkin_strategies]
        results   = axelrod.Tournament(
            [player] + opponents, turns=200, repetitions=3
        ).play(progress_bar=False)
        score = float(np.mean(results.normalised_scores[0]))
        return score, n_active
    except Exception as e:
        print(f"  [evaluate ERROR] {type(e).__name__}: {e}")
        return 0.0, n_active

In [ ]:
class FuzzyRuleProblem(Problem):

    def __init__(self):
        super().__init__(
            n_var=len(ALL_RULES),
            n_obj=2,
            vtype=bool
        )

    def _evaluate(self, X, out, *args, **kwargs):
        f1, f2 = [], []
        for row in X:
            n_active = int(sum(row))
            if n_active > 8 or n_active == 0:
                f1.append(0.0)
                f2.append(n_active)
                continue
            score, n_rules = evaluate_binary(row.astype(int))
            f1.append(-score)
            f2.append(n_rules)
        out["F"] = np.column_stack([f1, f2])

In [7]:
class SparseBinarySampling(Sampling):
    def _do(self, problem, n_samples, **kwargs):
        X = np.zeros((n_samples, problem.n_var), dtype=bool)
        for i in range(n_samples):
            n_active = np.random.randint(2, 9)
            active_indices = np.random.choice(problem.n_var, n_active, replace=False)
            X[i, active_indices] = True
        return X

In [ ]:
problem   = FuzzyRuleProblem()
algorithm = NSGA2(
    pop_size=20,
    sampling=SparseBinarySampling(),   # ← ovo
    crossover=UniformCrossover(),
    mutation=BitflipMutation(prob=0.02),
    eliminate_duplicates=True,
)

result = minimize(
    problem,
    algorithm,
    termination=('n_gen', 20),
    seed=1,
    verbose=True,
)

scores  = -result.F[:, 0]
n_rules =  result.F[:, 1]

print("\n=== PARETO FRONT ===")
pareto = sorted(zip(scores, n_rules, result.X), key=lambda x: -x[0])
for score, nr, binary in pareto:
    active = [ALL_RULES[i] for i, v in enumerate(binary) if v]
    print(f"{score:>8.4f} | {int(nr):>12} | {active}")

Scatter(title="Pareto Front: Skor vs Broj pravila").add(result.F).show()

best_idx    = np.argmax(scores)
best_binary = result.X[best_idx]
best_rules  = [ALL_RULES[i] for i, v in enumerate(best_binary) if v]

print(f"\n=== NAJBOLJI SKOR ===")
print(f"Skor: {scores[best_idx]:.4f} | Broj pravila: {int(n_rules[best_idx])}")
for r in best_rules:
    print(f"  adap={r[0]} coop={r[1]} forg={r[2]} stoch={r[3]} → {r[4]}")

n_gen  |  n_eval  | n_nds  |      eps      |   indicator  
     1 |       20 |      3 |             - |             -
     2 |       40 |      6 |  0.2857142857 |         ideal
     3 |       60 |      5 |  0.0010072138 |             f
     4 |       80 |      6 |  0.0014745247 |             f
     5 |      100 |      6 |  0.0014745247 |             f
     6 |      120 |      5 |  0.0016605417 |             f
     7 |      140 |      5 |  0.0016605417 |             f
     8 |      160 |      6 |  0.0020870196 |             f
     9 |      180 |      6 |  0.0020870196 |             f
    10 |      200 |      3 |  2.5000000000 |         nadir
    11 |      220 |      3 |  0.000000E+00 |             f
    12 |      240 |      3 |  0.000000E+00 |             f
    13 |      260 |      3 |  0.000000E+00 |             f
    14 |      280 |      4 |  0.0028478438 |         ideal
    15 |      300 |      4 |  0.000000E+00 |             f
    16 |      320 |      4 |  0.000000E+00 |            